# torchisotonic on a GPU

Compiles the CUDA operator, checks it against the CPU one (which the tests check against
scipy and scikit-learn), and benchmarks it against scipy and torchsort.

**Before running:** Settings → Accelerator → **GPU T4 x2**, Settings → **Internet on**. Then
**Run All in the editor**, so every cell's output shows up as it runs.

In [ ]:
!nvidia-smi

## Install

Uses a Kaggle dataset named `torchisotonic` if one is attached (an upload of the project
folder), otherwise clones the GitHub repository. Prints the commit or the dataset path, so
you know what is being tested.

In [ ]:
import os

REPO = "https://github.com/LenaBarretta/torchisotonic"
DATASET = "/kaggle/input/torchisotonic"
SRC = "/kaggle/working/torchisotonic"

!rm -rf {SRC}
if os.path.isdir(DATASET):
    !cp -r {DATASET} {SRC} && echo "from dataset {DATASET}"
else:
    !git clone -q {REPO} {SRC} && git -C {SRC} log -1 --format='%h %s'
!pip install -q pytest pytest-cov hypothesis scipy scikit-learn ninja
!pip install -q --no-deps --force-reinstall {SRC}

In [ ]:
import time, torch, torchisotonic
from torch.utils import cpp_extension
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))
print("CUDA_HOME", cpp_extension.CUDA_HOME)
start = time.perf_counter()
print(torchisotonic.isotonic(torch.tensor([[5., 3, 1, 2, 8, 10, 7, 9, 6, 4]], device="cuda")))
print(f"first call, including compiling the operators: {time.perf_counter() - start:.0f} s")

Optional: torchsort, the only other GPU isotonic regression (inside its soft sort). Its pip
install compiles a CUDA extension, which takes a few minutes; skip this cell to benchmark
without it.

In [ ]:
!pip install -q torchsort

## Correctness on the GPU

The full test suite: the CPU tests against scipy and scikit-learn, and `tests/test_cuda.py`,
which compares the GPU results with the CPU ones.

In [ ]:
!cd {SRC} && python -m pytest -q -p no:cacheprovider --no-cov tests

## Benchmark

Decreasing, unweighted problems (all torchsort solves), float32, from a million problems of
16 points to one problem of ten million points. scipy runs one problem at a time in a loop.

In [ ]:
!cd {SRC} && python bench/bench.py

In [ ]:
from pathlib import Path
from IPython.display import Markdown

Markdown("\n\n".join(p.read_text() for p in sorted(Path(SRC, "bench", "results").glob("*.md"))))

## Keep the results

Download `bench-results.zip` from the Output panel on the right (after the cell below), or
just download this notebook with its outputs.

In [ ]:
!cd {SRC} && zip -qr /kaggle/working/bench-results.zip bench/results && ls -l /kaggle/working/bench-results.zip